# 11-bob. 11-BOB. SI ASOSIDA AUTENTIFIKATSIYA VA FIRIBGARLIKNI ANIQLASH
## Amaliy misol: tranzaksiya belgilari va xatti-harakat belgilari

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/11-bob/11-2_tranzaksiya-belgilari-va-xatti-harakat.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 11-bob.*

Quyidagi dasturda 3000 nafar mijozning 30 kunlik tranzaksiyalari sun'iy ravishda yaratiladi. Har bir mijozning o'z odatiy xarid summasi, onlayn to'lovlar ulushi va faol vaqti bor. Mijozlarning taxminan 2 foizi kartasi ma'lumotlarini firibgarlarga boy beradi: firibgar qisqa vaqt ichida bir necha to'lov amalga oshiradi. Biroq firibgar to'lovlari ham har doim keskin ajralib turmaydi: ba'zilari kichik summada, kunduzi va mijozning odatiy qurilmasidan amalga oshiriladi. Ikki model taqqoslanadi: faqat tranzaksiyaning o'z belgilari bilan va mijoz tarixidan hisoblangan xatti-harakat belgilari bilan. Model birinchi 20 kun ma'lumotlarida o'qitiladi va oxirgi 10 kunda sinaladi. Bank tahlilchilari kuniga faqat 20 ta eng shubhali tranzaksiyani tekshira oladi, deb faraz qilinadi. Dastur amaliy mashg'ulotning A11 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

rng = np.random.default_rng(7)
yozuvlar = []
for mijoz in range(3000):
    odatiy_summa = rng.lognormal(11.5, 0.6)                 # so'm: odatda ~100 ming atrofida
    onlayn_ulush, kunduzgi = rng.uniform(0.05, 0.5), rng.uniform(0.7, 0.98)
    for kun in range(30):
        for _ in range(rng.poisson(1.5)):
            soat = rng.uniform(8, 22) if rng.random() < kunduzgi else rng.uniform(0, 24)
            yozuvlar.append((mijoz, kun * 24 + soat, odatiy_summa * rng.lognormal(0, 0.7),
                             rng.random() < onlayn_ulush, rng.random() < 0.01, rng.random() < 0.03, 0))
    if rng.random() < 0.02:                                   # karta ma'lumotlari o'g'irlangan mijoz
        boshi = rng.uniform(24, 30 * 24 - 5)
        for k in range(rng.integers(2, 8)):                   # firibgar qisqa vaqtda bir necha to'lov qiladi
            yozuvlar.append((mijoz, boshi + k * rng.uniform(0.05, 0.6), odatiy_summa * rng.lognormal(1.0, 0.9),
                             rng.random() < 0.8, rng.random() < 0.3, rng.random() < 0.6, 1))

df = pd.DataFrame(yozuvlar, columns=["mijoz", "vaqt", "summa", "onlayn", "xorij", "yangi_qurilma", "firibgarlik"])
df = df.sort_values(["mijoz", "vaqt"]).reset_index(drop=True)
df["soat"] = df.vaqt % 24
df["kun"] = (df.vaqt // 24).astype(int)

# Xatti-harakat xususiyatlari: faqat o'tmishdagi ma'lumotlardan hisoblanadi
g = df.groupby("mijoz")
df["oldingi_odatiy"] = g.summa.transform(lambda s: s.shift().expanding().median())
df["summa_nisbati"] = df.summa / df.oldingi_odatiy
df["oxirgi_tolovdan_soat"] = g.vaqt.diff()
df["oxirgi_1_soatda"] = g.vaqt.transform(
    lambda t: np.array([((t.values[:i] > v - 1)).sum() for i, v in enumerate(t.values)]))

oddiy = ["summa", "soat", "onlayn", "xorij", "yangi_qurilma"]
xulq = oddiy + ["summa_nisbati", "oxirgi_tolovdan_soat", "oxirgi_1_soatda"]
oqitish, sinov = df[df.kun < 20], df[df.kun >= 20]          # vaqt bo'yicha ajratish
print(f"Tranzaksiyalar: {len(df)}, firibgarlik ulushi: {df.firibgarlik.mean():.2%}\n")

for nom, belgilar in [("Faqat tranzaksiya belgilari", oddiy), ("Xatti-harakat belgilari bilan", xulq)]:
    model = HistGradientBoostingClassifier(random_state=0).fit(oqitish[belgilar], oqitish.firibgarlik)
    s = sinov.assign(ball=model.predict_proba(sinov[belgilar])[:, 1])
    # Tahlilchilar kuniga faqat 20 ta eng shubhali tranzaksiyani tekshira oladi
    tekshirilgan = s.sort_values("ball", ascending=False).groupby("kun").head(20)
    print(f"{nom}:")
    print(f"  aniqlik-to'liqlik egri chizig'i ostidagi yuza: {average_precision_score(s.firibgarlik, s.ball):.2f}")
    print(f"  tekshirilgan signallar orasida firibgarlik: {tekshirilgan.firibgarlik.mean():.0%}, "
          f"topilgan firibgarliklar: {tekshirilgan.firibgarlik.sum()} / {s.firibgarlik.sum()}")

Tranzaksiyalar: 135466, firibgarlik ulushi: 0.21%



Faqat tranzaksiya belgilari:
  aniqlik-to'liqlik egri chizig'i ostidagi yuza: 0.10
  tekshirilgan signallar orasida firibgarlik: 18%, topilgan firibgarliklar: 35 / 108


Xatti-harakat belgilari bilan:
  aniqlik-to'liqlik egri chizig'i ostidagi yuza: 0.33
  tekshirilgan signallar orasida firibgarlik: 30%, topilgan firibgarliklar: 59 / 108


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Tranzaksiyalar: 135466, firibgarlik ulushi: 0.21%

Faqat tranzaksiya belgilari:
  aniqlik-to'liqlik egri chizig'i ostidagi yuza: 0.10
  tekshirilgan signallar orasida firibgarlik: 18%, topilgan firibgarliklar: 35 / 108
Xatti-harakat belgilari bilan:
  aniqlik-to'liqlik egri chizig'i ostidagi yuza: 0.33
  tekshirilgan signallar orasida firibgarlik: 30%, topilgan firibgarliklar: 59 / 108
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 11-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.